#  Cats vs Dogs — TensorFlow / Keras
**MobileNetV2 Transfer Learning | Train / Val / Test split**

## 1. Download Dataset from Kaggle
```bash
pip install kagglehub tensorflow pillow matplotlib scikit-learn
set KAGGLE_USERNAME=your_username
set KAGGLE_KEY=your_api_key
```

In [ ]:
!pip install kagglehub tensorflow pillow matplotlib scikit-learn


In [ ]:
import os, shutil, pathlib

KAGGLE_DATASET = "aleemaparakatta/cats-and-dogs-mini-dataset"
LOCAL_DATA_DIR = "data"

try:
    import kagglehub
    dl_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_DATASET))
    def find_root(base):
        for root, dirs, _ in os.walk(base):
            if "cat_set" in dirs or "dog_set" in dirs:
                return pathlib.Path(root)
        return base
    data_root = find_root(dl_path)
    if not os.path.exists(LOCAL_DATA_DIR):
        shutil.copytree(str(data_root), LOCAL_DATA_DIR)
    print(f"✓ Data ready at ./{LOCAL_DATA_DIR}")
except ImportError:
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    os.system(f"kaggle datasets download -d {KAGGLE_DATASET} --unzip -p {LOCAL_DATA_DIR}")

for cls in sorted(pathlib.Path(LOCAL_DATA_DIR).iterdir()):
    if cls.is_dir():
        print(f"  {cls.name}/  — {len(list(cls.glob('*.*')))} images")

## 2. Imports

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print(f"TensorFlow : {tf.__version__}")
print(f"GPUs       : {tf.config.list_physical_devices('GPU')}")

## 3. Configuration

In [ ]:
DATA_DIR   = "data"      # contains cat_set/ and dog_set/
OUTPUT_DIR = "outputs"
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 15
LR         = 1e-3
SEED       = 42
# Split ratios: 70% train | 15% val | 15% test
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.15

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Config ready.")

## 4. Three-Way Split — Train / Val / Test
Since `cat_set/` and `dog_set/` contain all images with no pre-existing split,
we create **train (70%) / val (15%) / test (15%)** splits manually so the test set
is never seen during training.

In [ ]:
import random

SPLIT_DIR = "data_split"   # organised train/val/test copy
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def build_split(src_dir, dst_dir, val_frac=0.15, test_frac=0.15, seed=42):
    """Copy images from flat class folders into train/val/test structure."""
    if pathlib.Path(dst_dir).exists():
        print(f"'{dst_dir}' already exists — skipping split.")
        return
    random.seed(seed)
    for cls_folder in sorted(pathlib.Path(src_dir).iterdir()):
        if not cls_folder.is_dir():
            continue
        files = [f for f in cls_folder.iterdir()
                 if f.suffix.lower() in EXTS]
        random.shuffle(files)
        n       = len(files)
        n_test  = int(n * test_frac)
        n_val   = int(n * val_frac)
        splits  = {
            "test" : files[:n_test],
            "val"  : files[n_test : n_test + n_val],
            "train": files[n_test + n_val:],
        }
        for split, flist in splits.items():
            dest = pathlib.Path(dst_dir) / split / cls_folder.name
            dest.mkdir(parents=True, exist_ok=True)
            for f in flist:
                shutil.copy(f, dest / f.name)
        print(f"  {cls_folder.name}: train={len(splits['train'])} "
              f"val={len(splits['val'])} test={len(splits['test'])}")
    print(f"✓ Split saved to '{dst_dir}'")

build_split(DATA_DIR, SPLIT_DIR, val_frac=VAL_SPLIT, test_frac=TEST_SPLIT)

## 5. Build tf.data Pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_ds(folder, shuffle=False):
    return tf.keras.utils.image_dataset_from_directory(
        folder, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        shuffle=shuffle, seed=SEED, label_mode="binary"
    )

train_ds = load_ds(os.path.join(SPLIT_DIR, "train"), shuffle=True)
val_ds   = load_ds(os.path.join(SPLIT_DIR, "val"))
test_ds  = load_ds(os.path.join(SPLIT_DIR, "test"))

CLASS_NAMES = train_ds.class_names   # ['cat_set', 'dog_set']
print(f"Classes : {CLASS_NAMES}")
print(f"Train batches : {len(train_ds)} | Val : {len(val_ds)} | Test : {len(test_ds)}")

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name="augmentation")

def pre_train(x, y): return preprocess_input(augment(x, training=True)), y
def pre_eval(x, y):  return preprocess_input(x), y

train_ds = train_ds.map(pre_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(pre_eval,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(pre_eval,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
print("✓ Pipeline ready.")

## 6. Preview Sample Images

In [ ]:
raw_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(SPLIT_DIR, "train"),
    image_size=IMG_SIZE, batch_size=9, shuffle=True, seed=SEED
)
plt.figure(figsize=(10, 10))
for images, labels in raw_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(CLASS_NAMES[int(labels[i])], fontsize=11)
        plt.axis("off")
plt.suptitle("Sample Training Images", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_images.png"), dpi=120)
plt.show()

## 7. Build Model

In [ ]:
def build_model():
    base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet")
    base.trainable = False
    inp  = keras.Input(shape=(*IMG_SIZE, 3))
    x    = base(inp, training=False)
    x    = layers.GlobalAveragePooling2D()(x)
    x    = layers.Dropout(0.4)(x)
    x    = layers.Dense(256, activation="relu")(x)
    x    = layers.Dropout(0.3)(x)
    out  = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inp, out)

model = build_model()
model.summary()
print(f"Trainable : {sum(np.prod(v.shape) for v in model.trainable_weights):,.0f}")

## 8. Compile & Train

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

cbs = [
    keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, "best_model.keras"),
        monitor="val_accuracy", save_best_only=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cbs)

hist_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with open(os.path.join(OUTPUT_DIR, "history.json"), "w") as f:
    json.dump(hist_dict, f, indent=2)
print("✓ Training complete.")

## 9. Plot Training History

In [ ]:
h  = history.history
ep = range(1, len(h["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Training History", fontsize=14, fontweight="bold")
axes[0].plot(ep, h["loss"], "b-o", label="Train"); axes[0].plot(ep, h["val_loss"], "r-o", label="Val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep, h["accuracy"], "b-o", label="Train"); axes[1].plot(ep, h["val_accuracy"], "r-o", label="Val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].set_ylim(0, 1)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_history.png"), dpi=150)
plt.show()

In [ ]:
best_model = keras.models.load_model(os.path.join(OUTPUT_DIR, "best_model.keras"))

# ── Overall metrics ──
test_loss, test_acc = best_model.evaluate(test_ds, verbose=1)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.2%}")

# ── Collect predictions ──
y_true, y_pred = [], []
for images, labels in test_ds:
    probs  = best_model.predict(images, verbose=0)
    preds  = (probs.squeeze() > 0.5).astype(int)
    y_true.extend(labels.numpy().astype(int).flatten())
    y_pred.extend(preds.flatten())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ── Confusion Matrix ──
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — Test Set", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

# ── Per-class Report ──
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))